In [49]:
from datasets import get_dataset_split_names, load_dataset
from tiktoken import encoding_for_model

In [80]:
dataset_name = "codelion/fineweb-edu-1B"
get_dataset_split_names("codelion/fineweb-edu-1B")
enc = encoding_for_model("gpt-2")

In [81]:
enc.n_vocab

50257

In [82]:
def tokenize(x):
    ids = enc.encode_ordinary(x["text"])                                                                                                        
    ids.append(enc.eot_token)          # document boundary <|endoftext|>                                                                             
    return {"input_ids": ids}

In [ ]:
import torch
from torch.utils.data import DataLoader

CONTEXT_SIZE = 2048
BLOCK_SIZE = CONTEXT_SIZE + 1  # input + target in one packed window

class PackedTokenDataset(torch.utils.data.IterableDataset):
    """Streams an already-tokenized HF IterableDataset and packs all token IDs
    into fixed-size windows of BLOCK_SIZE tokens.

    Each yielded item is an LM example: x of length CONTEXT_SIZE and y,
    the same window shifted left by one position.
    """

    def __init__(self, dataset, block_size: int):
        self.dataset = dataset
        self.block_size = block_size

    def __iter__(self):
        buffer: list[int] = []
        for data in self.dataset:
            buffer.extend(data["input_ids"])  # eot already appended by tokenize()
            # emit as many full windows as fit in the buffer
            while len(buffer) >= self.block_size:
                chunk = buffer[: self.block_size]
                buffer = buffer[self.block_size :]
                x = torch.tensor(chunk[:-1], dtype=torch.long)
                y = torch.tensor(chunk[1:], dtype=torch.long)
                yield x, y

N_SHARDS = 10

train_raw = load_dataset(dataset_name, split="train", streaming=True).shard(N_SHARDS, index=0, contiguous=False)
val_raw = load_dataset(dataset_name, split="train", streaming=True).shard(N_SHARDS, index=9, contiguous=False)

train_dataset = PackedTokenDataset(train_raw.map(tokenize), BLOCK_SIZE)
val_dataset = PackedTokenDataset(val_raw.map(tokenize), BLOCK_SIZE)

BATCH_SIZE = 8

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=0)

In [ ]:
x, y = next(iter(train_loader))
vx, vy = next(iter(val_loader))
print("train:", tuple(x.shape), " val:", tuple(vx.shape))

assert torch.equal(x[0, 1:], y[0, :-1])
assert torch.equal(vx[0, 1:], vy[0, :-1])

print(enc.decode(x[0][:20].tolist()))
print("---")
print(enc.decode(y[0][:20].tolist()))

train: (8, 2048)  val: (8, 2048)
Discover the cosmos! Each day a different image or photograph of our fascinating universe is featured, along with
---
 the cosmos! Each day a different image or photograph of our fascinating universe is featured, along with a


In [ ]:
from torch import nn

EMBEDDING_DIM = 256
QUERY_DIM = 64
KEY_DIM = 64
VALUE_DIM = 64

class GPT(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        token_embedding_table = nn.Embedding(enc.n_vocab, EMBEDDING_DIM)
        pos_embedding_table = nn.Embedding(CONTEXT_SIZE, EMBEDDING_DIM)

    def forward(self, x):
        # Here x is a 1 dim tensor of tokens
        